# Aula 02 — Estruturas de dados e funções

**Trilha:** Santander — Dados com Python e IA (DIO)
**Objetivo:** ir além do básico: entender *referência x cópia*, usar compreensões, dominar `dict`/`set` e escrever funções com parâmetros flexíveis — fechando com um mini-projeto de normalização e agregação testado por `assert`.

**Pré-requisito:** Aula 01 (`dio-01-python.ipynb`).

**Tempo estimado:** 60–90 min.


In [ ]:
from __future__ import annotations

print("Ambiente pronto")

## 1. Listas: fatiamento, cópia e armadilhas

Lista é **mutável** e **por referência**. `b = a` não copia: aponta para a mesma lista. Use `a.copy()` (raso) ou `copy.deepcopy` (aninhado) quando quiser independência.

Fatiamento devolve **nova lista**: `a[1:4]`, `a[:2]`, `a[-2:]`, `a[::2]`, `a[::-1]`.


In [ ]:
notas = [7.5, 9.0, 6.0, 8.5, 10.0]

print("primeiras 3:", notas[:3])
print("últimas 2:  ", notas[-2:])
print("de 2 em 2:  ", notas[::2])
print("invertida:  ", notas[::-1])

a = [1, 2, 3]
b = a
b.append(4)
print("a:", a, "| b:", b, "| mesmo objeto?", a is b)

c = a.copy()
c.append(5)
print("a depois do copy:", a)
print("c:", c)

## 2. Compreensões

Uma linha para transformar/filtrar. A forma geral é `[expressao for item in iteravel if condicao]`. A versão com parênteses cria um **gerador** (lazy, não ocupa memória).


In [ ]:
quadrados = [n**2 for n in range(1, 6)]
print("quadrados:", quadrados)

pares = [n for n in range(10) if n % 2 == 0]
print("pares:", pares)

# dict comprehension: preço -> produto
precos = {"notebook": 3500.0, "mouse": 80.0, "teclado": 150.0}
caros = {produto: preco for produto, preco in precos.items() if preco > 100}
print("caros:", caros)

# set comprehension: categorias únicas a partir de pares (produto, categoria)
itens = [("notebook", "hardware"), ("mouse", "periferico"), ("monitor", "hardware")]
categorias = {categoria for _, categoria in itens}
print("categorias:", categorias)

# gerador: soma sem materializar lista
soma = sum(n**2 for n in range(1, 101))
print("soma dos quadrados 1..100:", soma)

## 3. Dicionários

`get` evita `KeyError`; `setdefault` inicializa listas; `|` (3.9+) mescla sem alterar os originais. A ordem de inserção é preservada desde o 3.7.


In [ ]:
estoque = {"notebook": 2, "mouse": 10}

print("webcam:", estoque.get("webcam"), "| com default:", estoque.get("webcam", 0))

vendas_por_categoria: dict[str, list[str]] = {}
for produto, categoria in [
    ("notebook", "hardware"),
    ("mouse", "periferico"),
    ("monitor", "hardware"),
]:
    vendas_por_categoria.setdefault(categoria, []).append(produto)
print(vendas_por_categoria)

padrao = {"garantia": "12 meses", "frete": "grátis"}
personalizado = {"garantia": "24 meses"}
config = padrao | personalizado
print("config:", config)
print("padrao intacto:", padrao)

for produto, qtd in sorted(estoque.items(), key=lambda par: par[1], reverse=True):
    print(f"{produto:<10} {qtd:>3}")

## 4. Conjuntos e tuplas

`set` remove duplicatas e faz união/interseção/diferença em O(1) médio. `tuple` é imutável e serve para retornos múltiplos e chaves de dicionário.

Para remover duplicatas **mantendo a ordem**, `list(dict.fromkeys(itens))` é o truque canônico.


In [ ]:
tags_a = {"python", "dados", "sql"}
tags_b = {"dados", "ia", "python"}

print("união:       ", tags_a | tags_b)
print("interseção:  ", tags_a & tags_b)
print("diferença:   ", tags_a - tags_b)

bruto = ["python", "dados", "python", "ia", "dados"]
sem_duplicata = list(dict.fromkeys(bruto))
print("sem duplicata e ordenado:", sem_duplicata)


def minmax(valores: list[float]) -> tuple[float, float]:
    return min(valores), max(valores)


menor, maior = minmax([3.0, 9.5, 1.2])
print("min/max:", menor, maior)

primeiro, *resto = [10, 20, 30, 40]
print("primeiro:", primeiro, "| resto:", resto)

## 5. Funções: parâmetros flexíveis (e uma armadilha clássica)

- `*args` → tupla de posicionais extras.
- `**kwargs` → dicionário de nomeados extras.
- `*` sozinho força argumentos **keyword-only**.
- **Nunca** use lista/dict como valor padrão: ele é criado **uma vez** e compartilhado entre chamadas.


In [ ]:
def somar(*valores: float) -> float:
    return sum(valores)


def montar_pedido(produto: str, *, quantidade: int = 1, **extras: str) -> dict:
    pedido = {"produto": produto, "quantidade": quantidade}
    pedido.update(extras)
    return pedido


print(somar(1, 2, 3, 4))
print(montar_pedido("notebook", quantidade=2, garantia="24 meses"))


def adicionar_bug(item: str, lista: list[str] = []) -> list[str]:  # noqa: B006
    """Exemplo ERRADO: o default é compartilhado entre chamadas."""
    lista.append(item)
    return lista


print("bug 1:", adicionar_bug("a"))
print("bug 2:", adicionar_bug("b"))


def adicionar(item: str, lista: list[str] | None = None) -> list[str]:
    """Versão correta: cria a lista dentro da chamada."""
    if lista is None:
        lista = []
    lista.append(item)
    return lista


print("ok 1:", adicionar("a"))
print("ok 2:", adicionar("b"))

## 6. Funções de alta ordem

`sorted`/`min`/`max` aceitam `key`; `map`/`filter` aplicam funções; `lambda` serve para coisas curtas. Prefira função nomeada quando a lógica passar de uma expressão.


In [ ]:
palavras = ["Dados", "ia", "Python", "sql", "Machine Learning"]

print("por tamanho:", sorted(palavras, key=len))
print("case-insensitive:", sorted(palavras, key=str.lower))

tamanhos = list(map(len, palavras))
grandes = list(filter(lambda p: len(p) > 3, palavras))
print("tamanhos:", tamanhos)
print("grandes:", grandes)

# a chave pode ser uma tupla: primeiro por categoria, depois por nome
produtos = [("periferico", "mouse"), ("hardware", "notebook"), ("periferico", "teclado")]
print(sorted(produtos))

## 7. Mini-projeto: normalização e agregação

Quatro funções pequenas — e agora **de produção**, em `src/opencode_harness/`:

1. `slug(texto)` — remove acentos e símbolos, deixa minúsculo (`texto.py`).
2. `agrupar_por(itens, chave, normalizar=None)` — agrupa registros por um campo (`texto.py`).
3. `top_n(contagem, n)` — ranking por valor (`texto.py`).
4. `receita_por_categoria(vendas)` — junta tudo (`vendas.py`).

Os testes de borda estão em `tests/test_texto.py` e `tests/test_vendas.py`.

In [ ]:
from opencode_harness import agrupar_por, receita_por_categoria, slug, top_n

vendas = [
    {"categoria": "hardware", "produto": "notebook", "valor": 3500.0, "qtd": 2},
    {"categoria": "periferico", "produto": "mouse", "valor": 80.0, "qtd": 10},
    {"categoria": "hardware", "produto": "monitor", "valor": 1200.0, "qtd": 1},
    {"categoria": "Periférico", "produto": "teclado", "valor": 150.0, "qtd": 4},
]

In [ ]:
assert slug("Análise de Dados") == "analise-de-dados"
assert slug("  Python  ") == "python"
assert slug("Preço / Valor (R$)") == "preco-valor-r"
# o `+` da regex colapsa símbolos seguidos: não sai "a--b"
assert slug("A/B") == "a-b"

# sem normalizar, "Periférico" ficaria em um grupo separado de "periferico"
grupos = agrupar_por(vendas, "categoria", normalizar=slug)
assert set(grupos) == {"hardware", "periferico"}
assert len(grupos["hardware"]) == 2
assert len(grupos["periferico"]) == 2

receita = receita_por_categoria(vendas)
assert receita["hardware"] == 8200.0
assert receita["periferico"] == 1400.0

ranking = top_n(receita, n=1)
assert ranking[0][0] == "hardware"
assert round(ranking[0][1], 2) == 8200.0

try:
    agrupar_por([{"produto": "mouse"}], "categoria")
except KeyError as erro:
    assert "item 0" in str(erro)
else:
    raise AssertionError("chave ausente deveria levantar KeyError")

print("receita:", receita)
print("ranking:", ranking)
print("testes OK")

## 8. De funções soltas para biblioteca

Este passo já foi dado: as funções vivem em `src/opencode_harness/texto.py` (texto e agrupamento) e
`src/opencode_harness/vendas.py` (`receita_por_categoria`), com testes em `tests/`.

O notebook segue como demonstração; o módulo é o código reutilizável — e é ele que o CI testa.

## Conclusões

- referência x cópia explica metade dos bugs com listas;
- compreensões deixam a intenção explícita (e geradores economizam memória);
- `dict` é a estrutura central de agregação — `get`/`setdefault` evitam `KeyError`;
- `set` resolve deduplicação e comparação de conjuntos;
- `*args`/`**kwargs`/`keyword-only` dão flexibilidade sem perder legibilidade;
- **default mutável é armadilha**: inicialize dentro da função;
- normalizar texto (acentos, caixa, espaços) antes de agrupar é regra de ouro.

**Próximos passos**

1. ✅ `slug`, `agrupar_por`, `top_n` e `receita_por_categoria` agora vivem em `src/opencode_harness/`,
   com testes em `tests/test_texto.py` e `tests/test_vendas.py`.
2. Refazer o mini-projeto com `pandas.groupby` e comparar os números.
3. Abrir PR desta branch (`dio/02-estruturas`).

**Checklist de entrega**

- [x] Notebook executa do início ao fim sem erro
- [x] Todos os `assert` passam
- [ ] Funções do mini-projeto com type hints e docstring
- [ ] PR aberto com o template preenchido
